# Baseline - Lost Interpreter | AICC Round 5

This notebook implements a deterministic baseline interpreter for the Lost Interpreter AICC Round 5 task. It expects `train.csv` and `test.csv` to be in the same directory as the notebook and writes `submission.csv`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SUBMISSION_PATH = DATA_DIR / 'submission.csv'

for required_path in [TRAIN_PATH, TEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f'Loaded {TRAIN_PATH}: {train_df.shape}')
print(f'Loaded {TEST_PATH}: {test_df.shape}')
train_df.head()

Loaded train.csv: (20000, 3)
Loaded test.csv: (5000, 2)


,Unnamed: 0,program,output
0,0,OP5 13 ; OP7 -2 ; OP1 1 ; OP8 2 ; OP10 0 ; OP1...,-48
1,1,OP5 -14 ; OP12 11 ; OP8 2 ; OP4 -3 ; OP12 17 ;...,3
2,2,OP5 -14 ; OP5 -8 ; OP12 5 ; OP4 -4 ; OP6 -2 4 0 3,0
3,3,OP5 -8 ; OP6 3 4 4 0 -4 ; OP4 2,0
4,4,OP5 -14 ; OP9 4 ; OP3 6 ; OP5 6 ; OP12 18 ; OP...,49


## Interpreter

The discovered mapping uses one integer state variable `x`, a program counter, and single-use absolute jumps for `OP1` and `OP3`.

In [2]:
INT_RE = re.compile(r'[-+]?\d+')


def parse_program(program: str):
    instructions = []

    for raw_instruction in str(program).split(';'):
        raw_instruction = raw_instruction.strip()
        if not raw_instruction:
            continue

        parts = raw_instruction.split()
        opcode = parts[0]
        operands = [int(x) for x in parts[1:]]
        instructions.append((opcode, operands))

    return instructions


def run_program(program: str) -> int:
    instructions = parse_program(program)

    x = 0
    pc = 0
    used_jumps = set()
    n = len(instructions)

    while 0 <= pc < n:
        opcode, operands = instructions[pc]

        if opcode in {'OP1', 'OP3'}:
            if pc not in used_jumps:
                used_jumps.add(pc)
                pc = operands[0]
            else:
                pc += 1
        elif opcode in {'OP2', 'OP9'}:
            pc += 1
        elif opcode == 'OP4':
            x = x // 2
            pc += 1
        elif opcode == 'OP5':
            x = operands[0]
            pc += 1
        elif opcode == 'OP6':
            product = 1
            for value in operands:
                product *= value
            x = product
            pc += 1
        elif opcode == 'OP7':
            x = x - operands[0]
            pc += 1
        elif opcode == 'OP8':
            x = x * operands[0]
            pc += 1
        elif opcode == 'OP10':
            x = x * operands[0] + 4
            pc += 1
        elif opcode == 'OP11':
            x = operands[0] * operands[1]
            pc += 1
        elif opcode == 'OP12':
            x = x + operands[0]
            pc += 1
        else:
            raise ValueError(f'Unknown opcode: {opcode}')

    return int(x)


def predict_programs(programs) -> np.ndarray:
    return np.array([run_program(program) for program in programs], dtype=object)

## Validate On Training Data

In [3]:
y_true = train_df['output'].astype(object).to_numpy()
y_pred = predict_programs(train_df['program'].astype(str))

absolute_errors = np.array(
    [abs(int(pred) - int(true)) for pred, true in zip(y_pred, y_true)],
    dtype=object,
)

mae = sum(absolute_errors) / len(absolute_errors)
mismatches = int(sum(error != 0 for error in absolute_errors))

print(f'Train MAE: {mae}')
print(f'Mismatches: {mismatches} / {len(train_df)}')

if mismatches:
    mismatch_indices = [i for i, error in enumerate(absolute_errors) if error != 0][:10]
    pd.DataFrame({
        'row': mismatch_indices,
        'predicted': [y_pred[i] for i in mismatch_indices],
        'actual': [y_true[i] for i in mismatch_indices],
        'program': [train_df.iloc[i]['program'] for i in mismatch_indices],
    })

Train MAE: 0.0
Mismatches: 0 / 20000


## Create Submission

In [4]:
test_predictions = predict_programs(test_df['program'].astype(str))

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'output': test_predictions,
})

submission.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved {SUBMISSION_PATH} with shape {submission.shape}')
submission.head()

Saved submission.csv with shape (5000, 2)


,ID,output
0,0,4
1,1,-6
2,2,0
3,3,0
4,4,29
